In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

ROOT = pathlib.Path().resolve().parent
sys.path.insert(0, str(ROOT / "src"))
sys.path.append('../../')

import pinn_torch as pinn
import const
import domain
import pde_residual
import synthetic_taylor_green
import plot_util
import json_util

plot_util.apply_plot_style()

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda:0


In [11]:
# Model Definition (2D Recast)
x_max, y_max, t_max = const.TG_X_RANGE[1], const.TG_Y_RANGE[1], const.TG_T_RANGE[1]
domain_sizes = [x_max, y_max, t_max]
num_centers = 11

# We use float64 for higher precision in PINNs
train_dtype = torch.float64

# 1. RBF layer as input to the neural network
rbf_layer = pinn.pinn.RadialBasisFunction(num_centers=num_centers, domain_sizes=domain_sizes)
# Ensure RBF buffers are float64
rbf_layer.rbf_centers = rbf_layer.rbf_centers.to(train_dtype)
rbf_layer.epsilon = rbf_layer.epsilon.to(train_dtype)

# 2. Backbone: Residual Fully Connected MLP
num_inputs = rbf_layer.rbf_centers.shape[1]
nodes_per_layer = 128
num_layers = 4
resnn = pinn.pinn.ResidualFullyConnected(num_inputs, num_layers, nodes_per_layer)

# 3. Output head: PhysicsLayer
# Fields are ("u", "v", "p") - 2D case
physics_layer = pinn.pinn.PhysicsLayer(num_inputs=nodes_per_layer, num_linear_outputs=3)

# 4. Composition: PhysicsHead
# Since Taylor-Green is time-dependent, we use PhysicsHead
net = pinn.pinn.PhysicsHead(
    layers=(rbf_layer, resnn, physics_layer),
    num_spatial=2 # x, y
)

net.theta = pinn.pinn.Theta()

# Learnable parameter for viscosity (inverse problem)
#net.theta.mu = nn.Parameter(torch.tensor([const.TG_MU_INIT], dtype=train_dtype))
net.theta.register_parameter(
    'mu',
    nn.Parameter(torch.tensor([const.TG_MU_INIT], dtype=train_dtype))
)

mse_cost_function = torch.nn.MSELoss()
groups = [
    {"params": list(net.stack.parameters()),"lr":1e-3},
    {"params": list(net.theta.parameters()),"lr":2e-2}
]
optimizer = torch.optim.Adam(groups)

net.to(device=device, dtype=train_dtype)
print(net)

PhysicsHead(
  (stack): Sequential(
    (0): RadialBasisFunction()
    (1): ResidualFullyConnected(
      (resnn): Sequential(
        (0): Linear(in_features=1331, out_features=128, bias=True)
        (1): Tanh()
        (2): LinearSkip(
          (linear): Linear(in_features=128, out_features=128, bias=True)
          (act): Tanh()
        )
        (3): LinearSkip(
          (linear): Linear(in_features=128, out_features=128, bias=True)
          (act): Tanh()
        )
        (4): Linear(in_features=128, out_features=128, bias=True)
      )
    )
    (2): PhysicsLayer(
      (linear_segment): Linear(in_features=128, out_features=3, bias=True)
    )
  )
  (theta): Theta()
)


In [4]:
[names for names,x in net.stack.named_parameters()]

['1.resnn.0.weight',
 '1.resnn.0.bias',
 '1.resnn.2.linear.weight',
 '1.resnn.2.linear.bias',
 '1.resnn.3.linear.weight',
 '1.resnn.3.linear.bias',
 '1.resnn.4.weight',
 '1.resnn.4.bias',
 '2.linear_segment.weight',
 '2.linear_segment.bias']

In [12]:
[names for names,x in net.theta.named_parameters()]

['mu']

In [6]:
class TaylorGreen:
    def __init__(self, mu_true=const.TG_MU_TRUE, rho=const.TG_RHO):
        self.mu_true = mu_true
        self.rho = rho
        self.fields = ("u", "v", "p")
        
    def f(self, model, spatial, t):
        # Combine coordinates for the model
        # model expects a list [X, Y, T]
        U = model(spatial, t)
        
        # Split output into fields
        u = U[:, 0:1]
        v = U[:, 1:2]
        p = U[:, 2:3]
        
        # Combine into a single coordinate list for differentiation
        coords = spatial + [t]
        
        # Compute derivatives
        u_diffs = pinn.differentiation.del_scalar(u, coords) # ux, uy, ut
        v_diffs = pinn.differentiation.del_scalar(v, coords) # vx, vy, vt
        p_diffs = pinn.differentiation.del_scalar(p, coords) # px, py, pt
        
        u_lap = pinn.differentiation.div_vector(u_diffs, coords)
        v_lap = pinn.differentiation.div_vector(v_diffs, coords)
        
        # Navier-Stokes residuals (2D)
        # Continuity: ux + vy = 0
        continuity = u_diffs[0] + v_diffs[1]
        
        # Momentum
        mu = model.theta.mu
        rho = self.rho
        
        # momentum_u: rho*(ut + u*ux + v*uy) + px - mu*u_lap
        mom_u = rho * (u_diffs[2] + u*u_diffs[0] + v*u_diffs[1]) + p_diffs[0] - mu*u_lap
        mom_v = rho * (v_diffs[2] + u*v_diffs[0] + v*v_diffs[1]) + p_diffs[1] - mu*v_lap
        
        return {
            "continuity": continuity,
            "momentum_u": mom_u,
            "momentum_v": mom_v
        }

    def get_data(self, n_samples, noise_std=0.0, seed=const.SEED):
        # Use existing 3D sampler and slice out the 2D components
        dataset = synthetic_taylor_green.sample_synthetic_dataset(
            n_samples=n_samples, mu=self.mu_true, rho=self.rho,
            x_range=const.TG_X_RANGE, y_range=const.TG_Y_RANGE, z_range=const.TG_Z_RANGE,
            t_range=const.TG_T_RANGE, noise_std=noise_std, seed=seed)
        
        # Remove z column from X (keep x, y, t)
        X = dataset["X"]
        dataset["X"] = np.hstack([X[:, 0:2], X[:, 3:4]])
        
        # Remove w field
        if "w" in dataset:
            del dataset["w"]
            
        return dataset

tg_prob = TaylorGreen()

In [7]:
# Data Setup
n_samples = const.TG_N_SAMPLES
dataset = tg_prob.get_data(n_samples=n_samples)

# Tensors for training
X_data = torch.tensor(dataset["X"], dtype=torch.float64, device=device).requires_grad_(True)
u_data = torch.tensor(dataset["u"], dtype=torch.float64, device=device)
v_data = torch.tensor(dataset["v"], dtype=torch.float64, device=device)
p_data = torch.tensor(dataset["p"], dtype=torch.float64, device=device)

# Split X into spatial and temporal parts
# X is now (x, y, t)
spatial_data = [X_data[:, i:i+1] for i in range(2)]
t_data = X_data[:, 2:3]

print(f"Dataset: {n_samples} samples")
print(f"Spatial shape: {spatial_data[0].shape}, Temporal shape: {t_data.shape}")

Dataset: 3000 samples
Spatial shape: torch.Size([3000, 1]), Temporal shape: torch.Size([3000, 1])


In [8]:
# Training Loop
iterations = 10000
collocation_size = 1000
losses = []

training_loss_string = '{epoch:6d} Training loss: {loss:g} | mu: {mu:.5g}'

for epoch in range(iterations):
    optimizer.zero_grad()
    
    # 1. Data Loss
    pred = net(spatial_data, t_data)
    mse_data = torch.mean(
        (pred[:, 0:1] - u_data)**2 + 
        (pred[:, 1:2] - v_data)**2 + 
        (pred[:, 2:3] - p_data)**2
    )
    
    # 2. PDE Loss
    # Sample collocation points (2D spatial: x, y)
    x_col = torch.tensor(np.random.uniform(low=0.0, high=1.0, size=(collocation_size, 2)), 
                         dtype=torch.float64, device=device).requires_grad_(True)
    t_col = torch.tensor(np.random.uniform(low=0.0, high=t_max, size=(collocation_size, 1)), 
                         dtype=torch.float64, device=device).requires_grad_(True)
    
    # The problem class expects spatial as a list
    spatial_col = [x_col[:, i:i+1] for i in range(2)]
    residuals = tg_prob.f(net, spatial_col, t_col)
    
    mse_pde = sum(torch.mean(res**2) for res in residuals.values())
    
    # Total Loss
    loss = mse_data + mse_pde
    
    loss.backward()
    optimizer.step()
    
    if epoch % 100 == 0:
        with torch.no_grad():
            current_loss = loss.item()
            current_mu = net.theta.mu.item()
            print(training_loss_string.format(epoch=epoch, loss=current_loss, mu=current_mu))
            losses.append((epoch, current_loss, current_mu))

print("Training complete.")

     0 Training loss: 1.32318 | mu: 0.98
   100 Training loss: 0.465153 | mu: 0.77196
   200 Training loss: 0.158433 | mu: 0.76197
   300 Training loss: 0.0642914 | mu: 0.74921
   400 Training loss: 0.0316276 | mu: 0.74193
   500 Training loss: 0.0214373 | mu: 0.7339
   600 Training loss: 0.0171911 | mu: 0.72402
   700 Training loss: 0.0411302 | mu: 0.71055
   800 Training loss: 0.0116108 | mu: 0.69608
   900 Training loss: 0.00989788 | mu: 0.68696
  1000 Training loss: 0.00888847 | mu: 0.67514
  1100 Training loss: 0.00813835 | mu: 0.658
  1200 Training loss: 0.00953008 | mu: 0.63945
  1300 Training loss: 0.00660819 | mu: 0.62943
  1400 Training loss: 0.00614798 | mu: 0.60999
  1500 Training loss: 0.00747606 | mu: 0.59693
  1600 Training loss: 0.00594614 | mu: 0.58535
  1700 Training loss: 0.00517885 | mu: 0.57333
  1800 Training loss: 0.00460809 | mu: 0.54537
  1900 Training loss: 0.00408575 | mu: 0.53891
  2000 Training loss: 0.00577977 | mu: 0.5258
  2100 Training loss: 0.00399622 

In [9]:
U

NameError: name 'U' is not defined

In [ ]:
U.shape

In [ ]:
U[0]

In [ ]:
UU = U.reshape(y_mesh,x_mesh,t_mesh,3).detach().cpu().numpy()
UU.shape

In [ ]:
UU[...,0,0].shape

In [ ]:
UU[:,:,:,:].shape

In [ ]:
x_max,y_max,t_max

In [ ]:
XX.shape

In [ ]:
xgrid.shape

In [ ]:
tgrid

In [1]:
# Visualization
x_mesh = 91
y_mesh = 81
t_mesh = 4
xgrid = np.linspace(0, x_max, x_mesh, dtype=np.float32)
ygrid = np.linspace(0, y_max, y_mesh, dtype=np.float32)
tgrid = np.linspace(0, t_max, t_mesh, dtype=np.float32)

XX, YY, TT = np.meshgrid(xgrid, ygrid, tgrid)
X_test = torch.tensor(XX.reshape(-1, 1), device=device).float()
Y_test = torch.tensor(YY.reshape(-1, 1), device=device).float()
T_test = torch.tensor(TT.reshape(-1, 1), device=device).float()
spatial_test = [X_test, Y_test]

with torch.no_grad():
    U = net(spatial_test, T_test)
    UU = U.reshape(y_mesh,x_mesh,t_mesh,3).detach().cpu().numpy()

plt.imshow(UU[...,-1, 0,], cmap='RdBu_r') # Plotting just the 'u' component
plt.colorbar()
plt.title("Predicted u field")
plt.xlabel("x")
plt.ylabel("y")
plt.show()

# Plot loss and recovered viscosity
epochs, loss_vals, mu_vals = zip(*losses)
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs, loss_vals)
plt.yscale('log')
plt.title("Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.subplot(1, 2, 2)
plt.plot(epochs, mu_vals)
plt.axhline(const.TG_MU_TRUE, color='orange', ls='--', label='True $\mu$')
plt.title("Recovered Viscosity $\mu$")
plt.xlabel("Epoch")
plt.ylabel("$\mu$")
plt.legend()

plt.tight_layout()
plt.show()

NameError: name 'np' is not defined

# 1. Synthetic Data

# 2. Train PINN

# 3. Noise Sensitivity

# 4. Sample Number Sensitivity